In [ ]:
import pandas as pd
from collections import defaultdict
import numpy as np
import gender_guesser.detector as gender
# Assuming you might want to display the final DataFrame later
from IPython.display import display, Markdown

def load_participants(csv_path):
    """
    Loads participants from CSV, filters for participation, removes duplicates by Name,
    and infers gender.
    """
    df = pd.read_csv(csv_path)
    # df.append()
    # Filter for participants who opted in
    df = df[df['Do you want to take part in the RL challenge?'] == 'Yes'].copy() # Use .copy() to avoid SettingWithCopyWarning

    # --- CORRECTION: Remove duplicate participants based on Name ---
    # Keep the first instance of each name found. Adjust 'subset' if a better unique ID exists (e.g., 'Email')
    initial_count = len(df)
    df = df.drop_duplicates(subset=['Name'], keep='first')
    duplicates_removed = initial_count - len(df)
    if duplicates_removed > 0:
        print(f"Removed {duplicates_removed} duplicate participant entries based on 'Name'.")
    # --------------------------------------------------------------

    # Infer gender (handle potential errors if name is not a string or empty)
    d = gender.Detector()
    df['sex'] = df['Name'].apply(lambda name: d.get_gender(str(name).split()[0]) if pd.notna(name) and str(name).strip() else 'unknown')

    return df

def compute_diversity_score(group):
    """
    Compute a simple diversity score based on counts of affiliation and inferred sex.
    Higher score means more diversity.
    """
    if not isinstance(group, pd.DataFrame): # Convert list of dicts to DataFrame if needed
        group = pd.DataFrame(group)
    if len(group) <= 1:
        return 0

    # Handle cases where columns might be missing or have no data
    affiliation_entropy = 0
    if 'Affiliation' in group.columns and not group['Affiliation'].empty:
        affiliations = group['Affiliation'].value_counts(normalize=True)
        affiliation_entropy = -sum(p * np.log(p) for p in affiliations if p > 0)

    sex_entropy = 0
    if 'sex' in group.columns and not group['sex'].empty:
        sexes = group['sex'].value_counts(normalize=True)
        sex_entropy = -sum(p * np.log(p) for p in sexes if p > 0)

    return affiliation_entropy + sex_entropy

def assign_groups(df, n_groups):
    """
    Assigns participants to groups, ensuring each participant is assigned only once.
    Balances based on RL experience, then greedily assigns based on coding experience.
    """
    # Ensure required columns exist
    required_cols = ['Name', 'Affiliation', 'sex',
                     'Rate your RL experience (On an increasing scale of 1-5)',
                     'Rate your coding experience (On an increasing scale of 1-5)']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in DataFrame: {col}")

    # Handle potential NaN values in rating columns before comparison
    df['Rate your RL experience (On an increasing scale of 1-5)'] = df['Rate your RL experience (On an increasing scale of 1-5)'].fillna(0)
    df['Rate your coding experience (On an increasing scale of 1-5)'] = df['Rate your coding experience (On an increasing scale of 1-5)'].fillna(0)


    # Separate participants based on RL experience.
    # Ensure the comparison is done with numeric types
    rl_exp_col = 'Rate your RL experience (On an increasing scale of 1-5)'
    high_rl = df[pd.to_numeric(df[rl_exp_col], errors='coerce') >= 3].copy()
    low_rl = df[pd.to_numeric(df[rl_exp_col], errors='coerce') < 3].copy() # Adjusted < 3 to be complementary to >= 3

    if len(high_rl) < n_groups:
        print(f"Warning: Only {len(high_rl)} high-experience participants available for {n_groups} groups. Some groups won't start with one.")
        # Adjust n_groups or handle this scenario as needed. For now, proceed but be aware.

    # Shuffle to randomize assignments. Use separate random states if desired.
    high_rl = high_rl.sample(frac=1, random_state=42).reset_index(drop=True)
    low_rl = low_rl.sample(frac=1, random_state=43).reset_index(drop=True) # Different random state

    groups = defaultdict(list)
    group_dfs = defaultdict(lambda: pd.DataFrame(columns=df.columns)) # Store as DataFrames
    coding_sums = np.zeros(n_groups)  # Track total coding experience per group

    # Convert coding experience to numeric, coercing errors to NaN, then fillna
    coding_exp_col = 'Rate your coding experience (On an increasing scale of 1-5)'
    high_rl[coding_exp_col] = pd.to_numeric(high_rl[coding_exp_col], errors='coerce').fillna(0)
    low_rl[coding_exp_col] = pd.to_numeric(low_rl[coding_exp_col], errors='coerce').fillna(0)


    # Start each group with one high RL participant, if available.
    num_high_assigned_initially = min(len(high_rl), n_groups)
    for i in range(num_high_assigned_initially):
        participant_series = high_rl.iloc[i]
        groups[i].append(participant_series.to_dict()) # Keep original dict format if needed elsewhere
        group_dfs[i] = pd.concat([group_dfs[i], participant_series.to_frame().T], ignore_index=True)
        coding_sums[i] += participant_series[coding_exp_col]

    # Remaining participants: rest of high RL and all low RL.
    remaining_high_rl_df = high_rl.iloc[num_high_assigned_initially:]
    all_remaining_df = pd.concat([remaining_high_rl_df, low_rl], ignore_index=True)

    # Sort remaining participants by coding experience descending for balanced distribution.
    all_remaining_df = all_remaining_df.sort_values(by=coding_exp_col, ascending=False)

    # Convert to list of dicts for iteration (original approach)
    remaining_participants_list = all_remaining_df.to_dict(orient='records')


    # Greedy assignment to the group with the lowest total coding experience.
    for participant_dict in remaining_participants_list:
        # Find the group index with the minimum coding sum
        # If multiple groups have the same minimum sum, np.argmin picks the first one.
        min_group_idx = np.argmin(coding_sums)

        groups[min_group_idx].append(participant_dict) # Keep dict format
        group_dfs[min_group_idx] = pd.concat([group_dfs[min_group_idx], pd.Series(participant_dict).to_frame().T], ignore_index=True)
        coding_sums[min_group_idx] += participant_dict[coding_exp_col]

    # Return the dictionary of lists of participant dictionaries OR dictionary of DataFrames
    # return groups  # Original return format
    return group_dfs # Return dict of DataFrames for easier diversity calculation etc.


def groups_to_dataframe(groups_dict):
    """
    Convert the groups dictionary (where keys are group indices and values are
    lists of participant dictionaries OR DataFrames) into a single DataFrame for display.
    Each column represents a group, listing the participant names.
    If groups have different numbers of participants, missing values will be NaN.
    """
    data = {}
    max_len = 0

    # Determine if input is dict of lists or dict of DataFrames
    first_value = next(iter(groups_dict.values())) if groups_dict else None
    is_dict_of_dfs = isinstance(first_value, pd.DataFrame)

    if is_dict_of_dfs:
        max_len = max(len(df) for df in groups_dict.values()) if groups_dict else 0
        for group_id, df in groups_dict.items():
            names = df['Name'].tolist()
            names += [None] * (max_len - len(names)) # Pad with None
            data[f'Group {group_id + 1}'] = names
    else: # Assuming dict of lists of dicts
        max_len = max(len(members) for members in groups_dict.values()) if groups_dict else 0
        for group_id, members in groups_dict.items():
            names = [member['Name'] for member in members]
            names += [None] * (max_len - len(names)) # Pad with None
            data[f'Group {group_id + 1}'] = names

    return pd.DataFrame(data)

In [6]:
print("Loading participants...")
participants_df = load_participants('registrations_all.csv')

print("\nParticipants after filtering and deduplication:")
display(participants_df[['Name', 'Affiliation', 'sex', 'Rate your RL experience (On an increasing scale of 1-5)']])

n_groups = 11
print(f"\nAssigning {len(participants_df)} participants to {n_groups} groups...")
try:
    # Use the version returning dict of DataFrames for easier analysis
    assigned_groups_dfs = assign_groups(participants_df.copy(), n_groups) # Pass a copy

    print("\nGroup Assignment Complete. Calculating diversity scores...")
    for i in range(n_groups):
         # Ensure the group exists before accessing
        if i in assigned_groups_dfs and not assigned_groups_dfs[i].empty:
             group_df = assigned_groups_dfs[i]
             diversity = compute_diversity_score(group_df) # Pass the DataFrame directly
             avg_coding = group_df['Rate your coding experience (On an increasing scale of 1-5)'].mean()
             avg_rl = group_df['Rate your RL experience (On an increasing scale of 1-5)'].mean()
             print(f"Group {i+1}: {len(group_df)} members, Avg Coding: {avg_coding:.2f}, Avg RL: {avg_rl:.2f}, Diversity Score: {diversity:.3f}")
             # Display group members (optional)
             # print(f"  Members: {group_df['Name'].tolist()}")
        else:
             print(f"Group {i+1}: 0 members (assignment might have failed or group is empty)")


    print("\nFinal Group Assignments (Names):")
    final_groups_df = groups_to_dataframe(assigned_groups_dfs)
    display(final_groups_df)

except ValueError as e:
    print(f"\nError during group assignment: {e}")
except Exception as e:
     print(f"\nAn unexpected error occurred: {e}")


Loading participants...

Participants after filtering and deduplication:


,Name,Affiliation,sex,Rate your RL experience (On an increasing scale of 1-5)
0,Konrad Altenmüller,ESS-BILBAO,male,1.0
2,Marco Bocchio,Transmutex,male,1.0
4,Ibon Bustinduy,ESS-BILBAO,male,2.0
7,Eya Dammak,Desy,unknown,1.0
8,Henry Day-Hall,DESY,male,4.0
9,Andre Dehne,HAW Hamburg,male,2.0
11,Auralee Edelen,SLAC,unknown,4.0
14,Ferdinand Ferber,University of Salzburg,male,1.0
15,Lorenz Fischl,MedAustron,male,2.0
16,Randeer Pratap Gautam,Universität Siegen,unknown,2.0



Assigning 43 participants to 11 groups...

Group Assignment Complete. Calculating diversity scores...
Group 1: 4 members, Avg Coding: 3.75, Avg RL: 1.75, Diversity Score: 1.099
Group 2: 4 members, Avg Coding: 3.75, Avg RL: 1.75, Diversity Score: 2.079
Group 3: 4 members, Avg Coding: 3.75, Avg RL: 2.25, Diversity Score: 1.949
Group 4: 4 members, Avg Coding: 3.75, Avg RL: 2.25, Diversity Score: 1.661
Group 5: 4 members, Avg Coding: 3.50, Avg RL: 1.50, Diversity Score: 2.079
Group 6: 4 members, Avg Coding: 3.50, Avg RL: 2.25, Diversity Score: 1.949
Group 7: 3 members, Avg Coding: 4.33, Avg RL: 2.00, Diversity Score: 1.735
Group 8: 4 members, Avg Coding: 3.75, Avg RL: 2.25, Diversity Score: 1.661
Group 9: 4 members, Avg Coding: 3.50, Avg RL: 2.50, Diversity Score: 1.949
Group 10: 4 members, Avg Coding: 3.50, Avg RL: 2.25, Diversity Score: 1.949
Group 11: 4 members, Avg Coding: 3.50, Avg RL: 1.75, Diversity Score: 2.426

Final Group Assignments (Names):


,Group 1,Group 2,Group 3,Group 4,Group 5,Group 6,Group 7,Group 8,Group 9,Group 10,Group 11
0,Georg Schäfer,Sabrina Pochaba,Henry Day-Hall,Parth Patil,Penny Madysa,Leander Grech,Auralee Edelen,Joel Wulff,M Asif Hasan,Olga Mironova,Hayg GULER
1,Marco Bocchio,Mohammad Sule Maidawa,Andre Dehne,Randeer Pratap Gautam,Johan Leygonie,Pardis Niknejadi,Julian Gethmann,Gerhard Hejc,Juan Luis Muñoz,Ibon Bustinduy,Amelia Pollard
2,Sebastian Starke,Ferdinand Ferber,Eya Dammak,Seyedeh Nasrin Mohammadi,Konrad Altenmüller,Farzad Jafarinia,Till Korten,Muhammad Abdullah Malik,Adrián Menor de Oñate,Jason St. John,Amna Majid
3,Hannes Voß,Gesa Goetzke,Matthew Schwab,Lorenz Fischl,Bindu Sharan,Stefano Krecic,None,Nadezhda Khachatrian,Francesco Tripaldi,Thorsten Hellert,Nicholas Tedjosantoso


In [7]:
# Assuming groups_df is your DataFrame with groups as columns:
md_table = final_groups_df.to_markdown(index=False)
display(Markdown(md_table))

| Group 1          | Group 2               | Group 3        | Group 4                  | Group 5            | Group 6          | Group 7         | Group 8                 | Group 9               | Group 10         | Group 11              |
|:-----------------|:----------------------|:---------------|:-------------------------|:-------------------|:-----------------|:----------------|:------------------------|:----------------------|:-----------------|:----------------------|
| Georg Schäfer    | Sabrina Pochaba       | Henry Day-Hall | Parth Patil              | Penny Madysa       | Leander Grech    | Auralee Edelen  | Joel Wulff              | M Asif Hasan          | Olga Mironova    | Hayg GULER            |
| Marco Bocchio    | Mohammad Sule Maidawa | Andre Dehne    | Randeer Pratap Gautam    | Johan Leygonie     | Pardis Niknejadi | Julian Gethmann | Gerhard Hejc            | Juan Luis Muñoz       | Ibon Bustinduy   | Amelia Pollard        |
| Sebastian Starke | Ferdinand Ferber      | Eya Dammak     | Seyedeh Nasrin Mohammadi | Konrad Altenmüller | Farzad Jafarinia | Till Korten     | Muhammad Abdullah Malik | Adrián Menor de Oñate | Jason St. John   | Amna Majid            |
| Hannes Voß       | Gesa Goetzke          | Matthew Schwab | Lorenz Fischl            | Bindu Sharan       | Stefano Krecic   |                 | Nadezhda Khachatrian    | Francesco Tripaldi    | Thorsten Hellert | Nicholas Tedjosantoso |